# 05 — Traduction en français + vérification (données scrapées + Wikidata)


On va faire ici deux choses en séquence :

1. Traduire les entités anglophones (RemoteOK, Arbeitnow, Wikidata) en français.
2. Valider chaque entité traduite (valid -> garder / reject -> supprimer).

In [1]:
import os

import csv, json, re
import jobkb_common as C
from collections import Counter, defaultdict


def load(path):
    if not os.path.isfile(path): return []
    with open(path, encoding="utf-8") as f: return list(csv.DictReader(f))

occs   = load(C.OCCUPATIONS_CSV)
skills = load(C.SKILLS_CSV)
labels = load(C.LABELS_CSV)

IO_DIR = os.path.join(C.ROOT, "llm_io")
PROMPT_DIR = os.path.join(IO_DIR, "prompts")
RESPONSE_DIR = os.path.join(IO_DIR, "responses")

for d in (PROMPT_DIR, RESPONSE_DIR):
    os.makedirs(d, exist_ok=True)
MAP_PATH = os.path.join(IO_DIR, "translation_index_map.json")

REVIEW_CSV = os.path.join(C.ROOT, "scraped", "review_translated.csv")

SCRAPED_SOURCES = {"REMOTEOK", "ARBEITNOW", "WIKIDATA"}

occ_todo = [o for o in occs if not o["pref_label_fr"].strip() and o["pref_label_en"].strip()]
skl_todo = [s for s in skills if not s["pref_label_fr"].strip()
            and s["pref_label_en"].strip() and s.get("hard_soft_provisional") != "group"]

print(f"À traduire — métiers : {len(occ_todo)}  | compétences : {len(skl_todo)}")
print("  métiers par source     :", dict(Counter(o["source"] for o in occ_todo)))
print("  compétences par source :", dict(Counter(s["source"] for s in skl_todo)))

À traduire — métiers : 168  | compétences : 127
  métiers par source     : {'WIKIDATA': 2, 'REMOTEOK': 39, 'ARBEITNOW': 127}
  compétences par source : {'REMOTEOK': 77, 'ARBEITNOW': 50}


## 5.1. Génération des prompts de traduction

In [2]:
BATCH = 100

index_map = {"occ_batches": [], "skl_batches": [], "created": C.now_iso()}

INSTRUCTIONS = (
    "Tu es un traducteur spécialisé dans les métiers et compétences de l'informatique (IT).\n"
    "Traduis en FRANÇAIS chaque libellé anglais ci-dessous.\n"
    "RÈGLES IMPORTANTES :\n"
    "- Garde TELS QUELS (sans traduire) les noms propres de technologies, langages, outils, "
    "frameworks et produits (ex. Python, Java, Kubernetes, Docker, PyTorch, Spring Boot, "
    "PostgreSQL, AWS, React). Ce sont des noms propres, on ne les traduit pas.\n"
    "- Traduis en revanche les INTITULÉS DE MÉTIERS et les compétences DESCRIPTIVES "
    "(ex. 'Software Developer' -> 'Développeur / Développeuse logiciel', "
    "'Machine Learning' -> 'Apprentissage automatique').\n"
    "- Utilise la terminologie française usuelle du secteur IT.\n"
    "- Si un libellé est déjà correct en français ou est un nom propre à conserver, "
    "renvoie-le inchangé.\n"
)

def make_prompt(batch, kind_letter):
    lines = "\n".join(f'{kind_letter}{i}: {e["pref_label_en"]}' for i, e in enumerate(batch))
    return (INSTRUCTIONS +
            f"\nLibellés à traduire (identifiants {kind_letter}0, {kind_letter}1, ...) :\n"
            f"{lines}\n\n"
            "Réponds STRICTEMENT en JSON, sans aucun texte autour, au format exact :\n"
            f'{{"translations": [{{"id": "{kind_letter}0", "fr": "..."}}]}}')

def write_batches(todo, kind_letter, prefix, map_key):
    n = 0
    for b in range(0, len(todo), BATCH):
        batch = todo[b:b + BATCH]
        index_map[map_key].append([e["entity_id"] for e in batch])
        path = os.path.join(PROMPT_DIR, f"{prefix}{b // BATCH:02d}.txt")
        with open(path, "w", encoding="utf-8") as f:
            f.write(make_prompt(batch, kind_letter))
        n += 1
    return n

n_occ = write_batches(occ_todo, "O", "trad_metiers_batch", "occ_batches")
n_skl = write_batches(skl_todo, "S", "trad_competences_batch", "skl_batches")

with open(MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(index_map, f, ensure_ascii=False, indent=1)

print(f"Prompts écrits : {n_occ} fichier(s) métiers + {n_skl} fichier(s) compétences")
print(f"Carte d'index -> {MAP_PATH}")

Prompts écrits : 2 fichier(s) métiers + 2 fichier(s) compétences
Carte d'index -> d:\JobKB-final\llm_io\translation_index_map.json


## 5.2. Traduction

Pour chaque fichier `trad_*.txt` dans `llm_io/prompts/` : copiez tout → collez dans un LLM → sauvez la réponse JSON dans `llm_io/responses/` (même nom de base).

In [3]:
def base_names(dir_):
    return {os.path.splitext(f)[0] for f in os.listdir(dir_)
            if f.endswith(".txt")} if os.path.isdir(dir_) else set()

trad_prompts = {n for n in base_names(PROMPT_DIR) if n.startswith("trad_")}
responses = base_names(RESPONSE_DIR)
print("État des réponses de traduction :")
for name in sorted(trad_prompts):
    mark = "✓" if name in responses else "…"
    print(f"  [{mark}] {name}")
missing = trad_prompts - responses
print(f"\n{len(trad_prompts) - len(missing)}/{len(trad_prompts)} avec réponse."
      if trad_prompts else "Aucun prompt de traduction.")

État des réponses de traduction :
  [✓] trad_competences_batch00
  [✓] trad_competences_batch01
  [✓] trad_metiers_batch00
  [✓] trad_metiers_batch01

4/4 avec réponse.


## 5.3. Ingérer les traductions

In [4]:
def extract_all_json(text):
    out = []
    fenced = re.findall(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, re.DOTALL)
    for blk in fenced:
        try: out.append(json.loads(blk))
        except Exception: pass
    if out: return out
    depth, start = 0, None
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                try: out.append(json.loads(text[start:i + 1]))
                except Exception: pass
                start = None
    return out

def read_response(name):
    p = os.path.join(RESPONSE_DIR, name + ".txt")
    if not os.path.isfile(p): return None
    with open(p, encoding="utf-8") as f: return f.read()

def tag_index(tag, prefix):
    m = re.fullmatch(prefix + r"(\d+)", str(tag).strip())
    return int(m.group(1)) if m else None

with open(MAP_PATH, encoding="utf-8") as f:
    index_map = json.load(f)

id2occ = {o["entity_id"]: o for o in occs}
id2skl = {s["entity_id"]: s for s in skills}

TRANS_FIELDS = ["translation_id", "entity_kind", "entity_id", "source",
                "label_en", "label_fr", "status", "reason", "created_at"]

def valid_translation(en, fr):
    fr = (fr or "").strip()
    if not fr: return False, "vide"
    if len(fr) > 200: return False, "trop long"
    if fr.lower() == en.strip().lower():
        return True, "nom propre conservé (non traduit)"
    return True, "ok"

suggestions, rejected = [], 0

def ingest(batches, prefix, kind, id2ent):
    global rejected
    for bi, batch_ids in enumerate(batches):
        txt = read_response(f"{prefix}{bi:02d}")
        if not txt: continue
        letter = "O" if kind == "occupation" else "S"
        for blk in extract_all_json(txt):
            for t in blk.get("translations", []):
                idx = tag_index(t.get("id", ""), letter)
                if idx is None or idx >= len(batch_ids):
                    rejected += 1; continue
                eid = batch_ids[idx]
                ent = id2ent.get(eid)
                if not ent:
                    rejected += 1; continue
                en = ent["pref_label_en"]
                fr = str(t.get("fr", "")).strip()
                ok, why = valid_translation(en, fr)
                if not ok:
                    rejected += 1; continue
                suggestions.append({
                    "translation_id": C.mint_id("TRAD_", eid, fr),
                    "entity_kind": kind, "entity_id": eid, "source": ent["source"],
                    "label_en": en, "label_fr": fr,
                    "status": "pending", "reason": why, "created_at": C.now_iso(),
                })

ingest(index_map.get("occ_batches", []), "trad_metiers_batch", "occupation", id2occ)
ingest(index_map.get("skl_batches", []), "trad_competences_batch", "skill", id2skl)

OUT = os.path.join(C.CANONICAL_DIR, "translation_suggestions.csv")
existing = load(OUT)
by_id = {r["translation_id"]: r for r in existing}
for s in suggestions: by_id[s["translation_id"]] = s
merged = list(by_id.values())
C.ensure_dirs()
with open(OUT, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=TRANS_FIELDS); w.writeheader()
    for r in merged: w.writerow({k: r.get(k, "") for k in TRANS_FIELDS})

print(f"{len(suggestions)} traduction(s) extraites, {rejected} rejetée(s).")
print(f"{len(merged)} au total dans translation_suggestions.csv.")

295 traduction(s) extraites, 0 rejetée(s).
295 au total dans translation_suggestions.csv.


## 5.4. Appliquer les traductions

In [5]:
trans = load(os.path.join(C.CANONICAL_DIR, "translation_suggestions.csv"))
to_apply = [t for t in trans if t.get("status") == "pending"]
print(f"Traductions à appliquer : {len(to_apply)}")

if to_apply:
    occ_by_id = {o["entity_id"]: o for o in occs}
    skl_by_id = {s["entity_id"]: s for s in skills}
    trans_by_entity = {t["entity_id"]: t for t in to_apply}

    def apply_to(entities, kind):
        touched = []
        for e in entities:
            t = trans_by_entity.get(e["entity_id"])
            if not t: continue
            fr = t["label_fr"].strip()
            en = e["pref_label_en"].strip()
            alts_en = [x for x in (e["alt_labels_en"].split(" | ") if e["alt_labels_en"] else []) if x]
            if en and en not in alts_en: alts_en.append(en)
            e["pref_label_fr"] = fr
            e["alt_labels_en"] = " | ".join(alts_en)
            e["label_language_status"] = "fr_translated"
            touched.append(e)
        return touched

    occ_touched = apply_to([o for o in occs if o["entity_id"] in trans_by_entity], "occupation")
    skl_touched = apply_to([s for s in skills if s["entity_id"] in trans_by_entity], "skill")

    for src in {e["source"] for e in occ_touched}:
        C.replace_source_rows(C.OCCUPATIONS_CSV, C.OCCUPATION_FIELDS, src,
                              [o for o in occs if o["source"] == src])
    for src in {e["source"] for e in skl_touched}:
        C.replace_source_rows(C.SKILLS_CSV, C.SKILL_FIELDS, src,
                              [s for s in skills if s["source"] == src])

    touched_sources = {e["source"] for e in occ_touched + skl_touched}
    all_labels = load(C.LABELS_CSV)
    for src in touched_sources:
        touched_ids = {e["entity_id"] for e in (occ_touched + skl_touched) if e["source"] == src}
        kept = [l for l in all_labels if l["source"] == src and l["entity_id"] not in touched_ids]
        new_labels = []
        for e in occ_touched + skl_touched:
            if e["source"] != src: continue
            kind = "occupation" if e["entity_id"] in occ_by_id else "skill"
            alts = {}
            if e["alt_labels_fr"]: alts["fr"] = [x for x in e["alt_labels_fr"].split(" | ") if x]
            if e["alt_labels_en"]: alts["en"] = [x for x in e["alt_labels_en"].split(" | ") if x]
            new_labels += C.make_label_rows(e["entity_id"], kind, src,
                                            preferred={"fr": [e["pref_label_fr"]]}, alts=alts)
        C.replace_source_rows(C.LABELS_CSV, C.LABEL_FIELDS, src, kept + new_labels)

    applied_ids = {t["translation_id"] for t in to_apply}
    for t in trans:
        if t["translation_id"] in applied_ids: t["status"] = "applied"
    with open(os.path.join(C.CANONICAL_DIR, "translation_suggestions.csv"),
              "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(trans[0].keys())); w.writeheader()
        for t in trans: w.writerow(t)

    print(f"Appliqué : {len(occ_touched)} métiers + {len(skl_touched)} compétences traduits.")
else:
    print("Rien à appliquer.")

Traductions à appliquer : 295
Appliqué : 168 métiers + 127 compétences traduits.


## 5.5. Revue de la traduction

Maintenant que les entités sont traduites, on génère un **fichier de revue** affichant le libellé **français** à côté de l'anglais. Vous marquez `valid` ou `reject` pour chaque ligne.

In [8]:
# Relire les données fraîches (post-traduction)
occs2 = load(C.OCCUPATIONS_CSV)
skills2 = load(C.SKILLS_CSV)

# Entités des sources scrapées/Wikidata (à valider)
occ_review = [o for o in occs2 if o["source"] in SCRAPED_SOURCES]
skl_review = [s for s in skills2 if s["source"] in SCRAPED_SOURCES
              and s.get("hard_soft_provisional") != "group"]

REVIEW_FIELDS = ["status", "kind", "source", "label_fr", "label_en", "entity_id", "notes"]

# Préserver les annotations existantes
existing_review = {}
if os.path.isfile(REVIEW_CSV):
    with open(REVIEW_CSV, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            existing_review[r.get("entity_id", "")] = r

review_rows = []
for o in occ_review:
    old = existing_review.get(o["entity_id"])
    review_rows.append({
        "status": old["status"] if old else "pending",
        "kind": "occupation", "source": o["source"],
        "label_fr": o["pref_label_fr"] or "(non traduit)",
        "label_en": o["pref_label_en"],
        "entity_id": o["entity_id"],
        "notes": old.get("notes", "") if old else "",
    })
for s in skl_review:
    old = existing_review.get(s["entity_id"])
    review_rows.append({
        "status": old["status"] if old else "pending",
        "kind": "skill", "source": s["source"],
        "label_fr": s["pref_label_fr"] or "(non traduit)",
        "label_en": s["pref_label_en"],
        "entity_id": s["entity_id"],
        "notes": old.get("notes", "") if old else "",
    })

os.makedirs(os.path.dirname(REVIEW_CSV), exist_ok=True)
with open(REVIEW_CSV, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=REVIEW_FIELDS)
    w.writeheader()
    for r in sorted(review_rows, key=lambda x: (x["kind"], x["source"], x["label_fr"])):
        w.writerow(r)

n_p = sum(1 for r in review_rows if r["status"] == "pending")
n_v = sum(1 for r in review_rows if r["status"] == "valid")
n_r = sum(1 for r in review_rows if r["status"] == "reject")
print(f"Fichier de revue : {REVIEW_CSV}")
print(f"  {sum(1 for r in review_rows if r['kind']=='occupation')} métiers | "
      f"{sum(1 for r in review_rows if r['kind']=='skill')} compétences")
print(f"  Statuts : {n_v} valid, {n_r} reject, {n_p} pending")
if n_p > 0:
    print(f"\n⚠ {n_p} entrée(s) en attente de validation.")
    print("Ouvrez le fichier, marquez 'valid' ou 'reject', puis ré-exécutez ce notebook.")
    print("Seules les entrées 'valid' resteront dans le graphe.")

Fichier de revue : d:\JobKB-final\scraped\review_translated.csv
  169 métiers | 127 compétences
  Statuts : 152 valid, 144 reject, 0 pending


## 5.6. Suppression des entités rejetées

Les entrées marquées `reject` sont **retirées** de la couche canonique (occupations, compétences, libellés, et liens associés). Les entrées `pending` (non encore validées) sont **conservées** en attendant votre décision. Seules les entrées `valid` restent dans le graphe.

In [9]:
# Relire la revue
reviewed = {}
if os.path.isfile(REVIEW_CSV):
    with open(REVIEW_CSV, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            reviewed[r["entity_id"]] = r.get("status", "pending")

rejected_ids = {eid for eid, st in reviewed.items() if st == "reject"}

if not rejected_ids:
    n_pending = sum(1 for st in reviewed.values() if st == "pending")
    if n_pending > 0:
        print(f"Aucune entrée rejetée. {n_pending} en attente de validation.")
        print("Complétez la revue puis ré-exécutez.")
    else:
        print("Aucune entrée rejetée. Toutes sont validées. ✓")
else:
    # Recharger les données fraîches
    occs3 = load(C.OCCUPATIONS_CSV)
    skills3 = load(C.SKILLS_CSV)
    labels3 = load(C.LABELS_CSV)
    rels3 = load(C.OCC_SKILL_REL_CSV)

    # Filtrer : retirer les entités rejetées et leurs liens
    occs_clean = [o for o in occs3 if o["entity_id"] not in rejected_ids]
    skills_clean = [s for s in skills3 if s["entity_id"] not in rejected_ids]
    labels_clean = [l for l in labels3 if l["entity_id"] not in rejected_ids]
    rels_clean = [r for r in rels3 if r["occupation_entity_id"] not in rejected_ids
                  and r["skill_entity_id"] not in rejected_ids]

    n_occ_rm = len(occs3) - len(occs_clean)
    n_skl_rm = len(skills3) - len(skills_clean)
    n_lab_rm = len(labels3) - len(labels_clean)
    n_rel_rm = len(rels3) - len(rels_clean)

    # Réécrire par source (idempotent)
    for src in SCRAPED_SOURCES:
        C.replace_source_rows(C.OCCUPATIONS_CSV, C.OCCUPATION_FIELDS, src,
                              [o for o in occs_clean if o["source"] == src])
        C.replace_source_rows(C.SKILLS_CSV, C.SKILL_FIELDS, src,
                              [s for s in skills_clean if s["source"] == src])
        C.replace_source_rows(C.LABELS_CSV, C.LABEL_FIELDS, src,
                              [l for l in labels_clean if l["source"] == src])
        C.replace_source_rows(C.OCC_SKILL_REL_CSV, C.REL_FIELDS, src,
                              [r for r in rels_clean if r["source"] == src])

    print(f"Entités rejetées supprimées du graphe :")
    print(f"  -{n_occ_rm} occupations, -{n_skl_rm} compétences, -{n_lab_rm} libellés, -{n_rel_rm} liens")

    # Lister ce qui a été rejeté
    print("\nDétail des rejets :")
    for eid in sorted(rejected_ids):
        row = next((r for r in review_rows if r["entity_id"] == eid), None)
        if row:
            print(f"  [{row['kind']:10s}] {row['label_fr']}  ({row['label_en']})")

    C.log_provenance("REVIEW_SCRAPED", [{
        "entity_id": "REVIEW", "source": "REVIEW_SCRAPED",
        "source_version": "revue manuelle post-traduction",
        "retrieved_at": C.now_iso(),
        "retrieval_method": "review_translated.csv, valid/reject",
        "notes": f"{len(rejected_ids)} rejetées, {n_occ_rm} occ/{n_skl_rm} skl supprimées",
    }])

Entités rejetées supprimées du graphe :
  -91 occupations, -53 compétences, -305 libellés, -661 liens

Détail des rejets :
  [occupation] audiologiste  (audiologist)
  [occupation] Conseiller fiscal / Conseillère fiscale (H/F/D) recherché(e) à Wiesloch, avec perspective d'association sur demande - à partir de 90 000 €  (Steuerberater (m/w/d) in Wiesloch, auf Wunsch mit Partnerperspektive, gesucht - mindestens 90.000€)
  [occupation] Directeur / Directrice de la réussite client  (Director of Customer Success)
  [occupation] Conseiller fiscal / Conseillère fiscale (H/F/D) recherché(e) à Glattbach, avec perspective d'association sur demande - à partir de 90 000 €  (Steuerberater (m/w/d) in Glattbach, auf Wunsch mit Partnerperspektive, gesucht - mindestens 90.000€)
  [occupation] Stagiaire Manager Artiste / Manager Influenceur (H/F/D)  (Trainee Artist Manager / Influencer Manager (m/w/d))
  [occupation] Testeur / Testeuse  (Tester:in)
  [occupation] Responsable marketing B2B IT / Services 

## 5.7. Vérifications

In [11]:
occs4 = load(C.OCCUPATIONS_CSV)
skills4 = load(C.SKILLS_CSV)
still_no_fr = [o for o in occs4 if not o["pref_label_fr"].strip() and o["pref_label_en"].strip()]
print("Après traduction + revue :")
print(f"  métiers sans fr : {len(still_no_fr)}")
print("  statuts occ :", dict(Counter(o["label_language_status"] for o in occs4)))
for src in sorted(SCRAPED_SOURCES):
    n = sum(1 for o in occs4 if o["source"] == src)
    print(f"  {src:10s} : {n} occupations dans le graphe")
if not still_no_fr:
    print("\n✓ Couverture française complète.")

Après traduction + revue :
  métiers sans fr : 0
  statuts occ : {'fr_native': 199, 'fr_translated': 78}
  ARBEITNOW  : 58 occupations dans le graphe
  REMOTEOK   : 19 occupations dans le graphe
  WIKIDATA   : 1 occupations dans le graphe

✓ Couverture française complète.
